# Qwen3-0.6B mixed SFT → one-epoch DPO

This notebook is designed for VS Code connected to a Google Colab A100 runtime. It starts from the final/best coherence-preserving mixed-SFT model and performs one full DPO epoch over the existing control bundle.

**Epoch clarification:** the earlier two-epoch experiment used all 4,500 training pairs in epoch 1 and then repeated those same 4,500 pairs in epoch 2. It did not divide the examples between epochs. This notebook therefore uses all 4,500 pairs exactly once.

Safeguards:

- starts from the final mixed-SFT weights;
- uses a gentler `5e-7` DPO learning rate;
- evaluates before training and near every quarter epoch;
- saves quarter-epoch checkpoints and the final model to Drive;
- reports training and held-out preference accuracy;
- supports checkpoint resumption.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Enable a Colab GPU runtime first."
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, f"({props.total_memory / 2**30:.1f} GiB)")
assert "A100" in props.name, "Select an A100 runtime for this full-DPO notebook."
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

## 1. Mount Drive, find the SFT model, and unpack the DPO bundle

The SFT output and training results remain on Drive. Copy `colab_qwen_dpo_bundle.zip` into the root of `MyDrive` if it is not already there.

In [ ]:
import shutil
import zipfile

from google.colab import drive

drive.mount("/content/drive")

SFT_MODEL = Path(
    "/content/drive/MyDrive/CoT_Controllability/"
    "qwen3-0.6b-control500-plus-svamp-sft-cutoff4096-1epoch"
)
BUNDLE = Path("/content/drive/MyDrive/colab_qwen_dpo_bundle.zip")
assert SFT_MODEL.is_dir(), f"SFT model directory is missing: {SFT_MODEL}"
assert (SFT_MODEL / "config.json").is_file(), f"Final SFT config is missing: {SFT_MODEL}"
assert any(SFT_MODEL.glob("*.safetensors")) or any(SFT_MODEL.glob("pytorch_model*.bin")), (
    "Final SFT weights are missing. Finish the SFT notebook or point SFT_MODEL "
    "to the selected complete SFT checkpoint."
)
assert BUNDLE.is_file(), f"Copy the prepared DPO bundle to {BUNDLE}"

PROJECT = Path("/content/qwen3_06b_mixed_sft_dpo")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(PROJECT)
(PROJECT / "data").mkdir(exist_ok=True)
(PROJECT / "scripts").mkdir(exist_ok=True)
for path in PROJECT.glob("*.json"):
    shutil.move(str(path), PROJECT / "data" / path.name)
for path in PROJECT.glob("*.py"):
    shutil.move(str(path), PROJECT / "scripts" / path.name)

SOURCE = PROJECT / "data/multilingual_thinking_qwen3_4b_cots.json"
PREPARE_DPO = PROJECT / "scripts/prepare_all_dpo_data.py"
assert SOURCE.is_file() and PREPARE_DPO.is_file()
print("Starting model:", SFT_MODEL)
print("Temporary workspace:", PROJECT)

## 2. Install LLaMA-Factory

This uses the same pinned stack as the earlier successful DPO run and preserves Colab's CUDA-enabled PyTorch.

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "llamafactory[torch,metrics]==0.9.5",
        "transformers>=4.51,<4.57",
        "datasets>=3.2",
        "accelerate>=1.2",
        "sentencepiece",
        "protobuf",
        "tensorboard",
        "matplotlib",
        "pandas",
        "pyyaml",
    ],
    check=True,
)
import llamafactory
import transformers
print("LLaMA-Factory:", llamafactory.__version__)
print("Transformers:", transformers.__version__)

## 3. Rebuild and validate all 5,000 preference pairs

There are 4,500 training pairs and 500 held-out validation pairs across three English controls and seven multilingual controls.

In [ ]:
import json

subprocess.run(
    [
        sys.executable,
        str(PREPARE_DPO),
        "--input", str(SOURCE),
        "--output-dir", str(PROJECT / "data"),
        "--eval-size", "100",
        "--seed", "42",
    ],
    check=True,
)

registry = json.loads((PROJECT / "data/dataset_info.json").read_text(encoding="utf-8"))
train_names = sorted(name for name in registry if name.endswith("_dpo_train"))
eval_names = sorted(name for name in registry if name.endswith("_dpo_eval"))
assert len(train_names) == len(eval_names) == 10

def registered_count(name):
    path = PROJECT / "data" / registry[name]["file_name"]
    return len(json.loads(path.read_text(encoding="utf-8")))

train_count = sum(registered_count(name) for name in train_names)
eval_count = sum(registered_count(name) for name in eval_names)
assert (train_count, eval_count) == (4500, 500), (train_count, eval_count)
print("Training datasets:", train_names)
print("Validation datasets:", eval_names)
print(f"One epoch exposure: {train_count} unique training pairs once")
print(f"Held-out validation: {eval_count} pairs")

## 4. Write the conservative one-epoch DPO configuration

With micro-batch one and accumulation four, one epoch is approximately 1,125 optimizer steps. Evaluation and saving occur near 25%, 50%, 75%, and 100% of that run.

In [ ]:
import math
import yaml

MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 4
EPOCHS = 1.0
EXPECTED_STEPS = math.ceil(train_count / (MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION))
QUARTER_STEPS = max(1, EXPECTED_STEPS // 4)
assert EXPECTED_STEPS == 1125

DRIVE_OUTPUT = Path(
    "/content/drive/MyDrive/CoT_Controllability/"
    "qwen3-0.6b-mixed-sft-then-controls-dpo-cutoff4096-1epoch"
)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

config = {
    "stage": "dpo",
    "do_train": True,
    "finetuning_type": "full",
    "pref_beta": 0.1,
    "pref_loss": "sigmoid",
    "model_name_or_path": str(SFT_MODEL),
    "dataset_dir": str(PROJECT / "data"),
    "dataset": ",".join(train_names),
    "eval_dataset": ",".join(eval_names),
    "template": "qwen3",
    "cutoff_len": 4096,
    "overwrite_cache": True,
    "preprocessing_num_workers": 4,
    "output_dir": str(DRIVE_OUTPUT),
    "overwrite_output_dir": False,
    "report_to": "tensorboard",
    "logging_steps": 1,
    "disable_tqdm": False,
    "plot_loss": True,
    "per_device_train_batch_size": MICRO_BATCH_SIZE,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION,
    "gradient_checkpointing": True,
    "bf16": True,
    "tf32": True,
    "learning_rate": 5.0e-7,
    "num_train_epochs": EPOCHS,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.05,
    "max_grad_norm": 1.0,
    "eval_strategy": "steps",
    "eval_steps": QUARTER_STEPS,
    "eval_on_start": True,
    "save_strategy": "steps",
    "save_steps": QUARTER_STEPS,
    "save_total_limit": 5,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_rewards/accuracies",
    "greater_is_better": True,
    "seed": 42,
    "data_seed": 42,
}

checkpoints = sorted(
    DRIVE_OUTPUT.glob("checkpoint-*"),
    key=lambda path: int(path.name.split("-")[-1]),
)
if checkpoints:
    config["resume_from_checkpoint"] = str(checkpoints[-1])
    print("Will resume from:", checkpoints[-1])

CONFIG_PATH = PROJECT / "mixed_sft_then_dpo_1epoch.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
(DRIVE_OUTPUT / "training_config.yaml").write_text(
    CONFIG_PATH.read_text(encoding="utf-8"), encoding="utf-8"
)
print("Expected optimizer steps:", EXPECTED_STEPS)
print("Evaluate/save every", QUARTER_STEPS, "steps")
print("Output:", DRIVE_OUTPUT)
print(CONFIG_PATH.read_text(encoding="utf-8"))

## 5. Train with a visible progress bar

The initial validation can take several minutes before the training bar begins. Re-running after an interruption resumes from the newest complete Drive checkpoint.

In [ ]:
import time

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"

stdout_path = DRIVE_OUTPUT / "training_stdout.log"
stderr_path = DRIVE_OUTPUT / "training_stderr.log"
progress_path = DRIVE_OUTPUT / "trainer_log.jsonl"
command = [sys.executable, "-m", "llamafactory.cli", "train", str(CONFIG_PATH)]

# Re-running this cell reconnects to a process started earlier in this kernel.
if "training_process" not in globals() or training_process.poll() is not None:
    training_stdout = stdout_path.open("a", encoding="utf-8")
    training_stderr = stderr_path.open("a", encoding="utf-8")
    training_process = subprocess.Popen(
        command,
        cwd=PROJECT,
        env=env,
        stdout=training_stdout,
        stderr=training_stderr,
    )
    print(f"Started DPO training PID {training_process.pid}", flush=True)
else:
    print(f"Reconnected to DPO training PID {training_process.pid}", flush=True)
print("Detailed log:", stderr_path, flush=True)
print("Waiting for model loading and initial validation...", flush=True)

last_signature = None
last_heartbeat = time.time()
while training_process.poll() is None:
    if progress_path.exists():
        lines = [
            line for line in progress_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        if lines:
            status = json.loads(lines[-1])
            signature = (
                status.get("current_steps"),
                status.get("eval_loss"),
                status.get("loss"),
            )
            if signature != last_signature:
                if "eval_loss" in status:
                    print(
                        f"VALIDATION | step {status.get('current_steps', status.get('step', 0))} | "
                        f"loss {status['eval_loss']:.4f} | "
                        f"preference accuracy "
                        f"{status.get('eval_rewards/accuracies', float('nan')):.1%}",
                        flush=True,
                    )
                else:
                    current = status.get("current_steps", status.get("step", 0))
                    total = status.get("total_steps", EXPECTED_STEPS)
                    percentage = status.get(
                        "percentage", 100 * current / total if total else 0
                    )
                    print(
                        f"TRAIN | {current}/{total} ({percentage:.2f}%) | "
                        f"loss {status.get('loss', float('nan')):.4f} | "
                        f"preference accuracy "
                        f"{status.get('accuracy', status.get('rewards/accuracies', float('nan'))):.1%} | "
                        f"elapsed {status.get('elapsed_time', '?')} | "
                        f"ETA {status.get('remaining_time', '?')}",
                        flush=True,
                    )
                last_signature = signature
                last_heartbeat = time.time()
    if time.time() - last_heartbeat >= 60:
        print("Still loading, validating, or waiting for the next trainer update...", flush=True)
        last_heartbeat = time.time()
    time.sleep(10)

if "training_stdout" in globals() and not training_stdout.closed:
    training_stdout.close()
if "training_stderr" in globals() and not training_stderr.closed:
    training_stderr.close()
if training_process.returncode != 0:
    tail = stderr_path.read_text(encoding="utf-8", errors="replace").splitlines()[-50:]
    raise RuntimeError("DPO training failed:\n" + "\n".join(tail))
print("DPO completed successfully.", flush=True)

## 6. Report training versus validation preference accuracy

Preference accuracy measures whether the preferred controlled response receives a higher reward than the rejected response. It is not ReasonIF answer accuracy; run the local ReasonIF notebook afterward before choosing this model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

state_candidates = list(DRIVE_OUTPUT.glob("checkpoint-*/trainer_state.json"))
if (DRIVE_OUTPUT / "trainer_state.json").is_file():
    state_candidates.append(DRIVE_OUTPUT / "trainer_state.json")
assert state_candidates, "No trainer_state.json was found."
state_path = max(state_candidates, key=lambda path: path.stat().st_mtime)
history = json.loads(state_path.read_text(encoding="utf-8"))["log_history"]

# Average training preference accuracy within each interval ending at an
# evaluation event. Baseline evaluation intentionally has no training value.
pending_train_accuracies = []
report_rows = []
for row in history:
    if "rewards/accuracies" in row:
        pending_train_accuracies.append(float(row["rewards/accuracies"]))
    if "eval_rewards/accuracies" in row:
        report_rows.append({
            "epoch": float(row.get("epoch", 0.0)),
            "step": int(row["step"]),
            "training_accuracy_since_previous_eval": (
                sum(pending_train_accuracies) / len(pending_train_accuracies)
                if pending_train_accuracies else float("nan")
            ),
            "training_logged_batches": len(pending_train_accuracies),
            "validation_accuracy": float(row["eval_rewards/accuracies"]),
            "validation_loss": float(row["eval_loss"]),
        })
        pending_train_accuracies = []

report = pd.DataFrame(report_rows)
assert not report.empty, "No DPO validation metrics were found."
report["generalization_gap"] = (
    report["training_accuracy_since_previous_eval"] - report["validation_accuracy"]
)
csv_path = DRIVE_OUTPUT / "quarter_train_vs_validation_accuracy.csv"
report.to_csv(csv_path, index=False)
display(report)

axis = report.plot(
    x="epoch",
    y=["training_accuracy_since_previous_eval", "validation_accuracy"],
    marker="o",
    ylim=(0, 1.02),
    title="DPO preference accuracy by training interval",
)
axis.set_ylabel("Preference accuracy")
axis.grid(alpha=0.25)
figure_path = DRIVE_OUTPUT / "quarter_train_vs_validation_accuracy.png"
axis.figure.tight_layout()
axis.figure.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
print("CSV:", csv_path)
print("Plot:", figure_path)

weight_files = list(DRIVE_OUTPUT.glob("*.safetensors")) + list(DRIVE_OUTPUT.glob("pytorch_model*.bin"))
saved_checkpoints = sorted(DRIVE_OUTPUT.glob("checkpoint-*"))
print("Final weight files:", [path.name for path in weight_files])
print("Checkpoints:", [path.name for path in saved_checkpoints])

## 7. TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/CoT_Controllability/qwen3-0.6b-mixed-sft-then-controls-dpo-cutoff4096-1epoch/runs

## 8. Release the Colab GPU when completely finished (optional)

In [ ]:
from google.colab import runtime
runtime.unassign()